# Assignment 8 - Midsem questions

    This notebook works through all five questions in the supplied Monsoon 2026 midsem paper. Each section states its convention, computes the answer in Python, and interprets the displayed result. No sentence boundary markers are added where the question excludes them.

## 1. Noun number finite-state transducer

    The transducer has two lexical paths. The regular path copies the noun stem and maps `+SG` to the empty string or `+PL` to `s`. The sibilant path is selected for stems ending in `s`, `x`, `z`, `ch`, or `sh`; it maps plural to `es`. Both paths emit `+N` as a grammatical feature on the lexical side and consume it without changing the surface form. The code models the suffix decision and prints representative singular/plural pairs. In an analyzer, the direction is reversed: a surface plural is mapped back to the stem plus `+PL+N`.

In [3]:
import re

SIBILANT_ENDINGS = ("ch", "sh", "s", "x", "z")

def plural_suffix(stem):
    return "es" if stem.lower().endswith(SIBILANT_ENDINGS) else "s"

def generate_noun(stem, number):
    if number == "+SG":
        return stem
    if number == "+PL":
        return stem + plural_suffix(stem)
    raise ValueError("number must be +SG or +PL")

examples = ["cat", "bus", "box", "buzz", "church", "dish"]
[(w, generate_noun(w, "+SG"), generate_noun(w, "+PL")) for w in examples]

[('cat', 'cat', 'cats'), ('bus', 'bus', 'buses'), ('box', 'box', 'boxes'), ('buzz', 'buzz', 'buzzes'), ('church', 'church', 'churches'), ('dish', 'dish', 'dishes')]

**How to read the output.** The ordinary path gives `cat/cats`; the special path gives `bus/buses`, `box/boxes`, `buzz/buzzes`, `church/churches`, and `dish/dishes`. These are sample realizations of the FST rule, not a claim that the simplified suffix rule covers every English irregular plural.

    **Transition table** (lexical input : surface output):

    | State | Input | Output | Next state |
    |---|---|---|---|
    | q0 | noun stem not ending in s/x/z/ch/sh | same stem | qR |
    | q0 | noun stem ending in s/x/z/ch/sh | same stem | qS |
    | qR | `+SG` | epsilon | qF |
    | qR | `+PL` | `s` | qF |
    | qS | `+SG` | epsilon | qF |
    | qS | `+PL` | `es` | qF |
    | qF | `+N` | epsilon | qAccept |

    `q0` branches based on the stem's final characters; `qF` is accepting after the grammatical noun feature. A finite lexicon would represent the stem copy as character-level identity transitions.

## 2. WordPiece with two merges

    WordPiece starts from character pieces and marks non-initial pieces with `##`. For merge learning, each word is represented as character units followed by `</w>`, weighted by its corpus frequency. At each iteration choose the adjacent pair maximizing `count(pair) / (count(left) * count(right))`, the WordPiece association score. Lexicographic order breaks ties.

In [6]:
from collections import Counter

corpus = Counter({"low": 5, "lower": 2, "lowest": 3, "new": 4, "newer": 2})

def initial_units(word):
    return tuple(word) + ("</w>",)

segmented = {word: initial_units(word) for word in corpus}
merge_history = []
for _ in range(2):
    pair_counts = Counter()
    unit_counts = Counter()
    for word, units in segmented.items():
        for unit in units:
            unit_counts[unit] += corpus[word]
        for pair in zip(units, units[1:]):
            pair_counts[pair] += corpus[word]
    scores = {pair: count / (unit_counts[pair[0]] * unit_counts[pair[1]])
              for pair, count in pair_counts.items()}
    top_score = max(scores.values())
    pair = sorted(pair for pair, score in scores.items() if score == top_score)[0]
    merged = "".join(pair)
    merge_history.append((pair, pair_counts[pair], top_score))
    for word, units in segmented.items():
        result, i = [], 0
        while i < len(units):
            if i + 1 < len(units) and units[i:i+2] == pair:
                result.append(merged)
                i += 2
            else:
                result.append(units[i])
                i += 1
        segmented[word] = tuple(result)

merge_history, segmented, segmented["new"] + ("e", "s", "t", "</w>")

([(('s', 't'), 3, 0.3333333333333333), (('l', 'o'), 10, 0.1)], {'low': ('lo', 'w', '</w>'), 'lower': ('lo', 'w', 'e', 'r', '</w>'), 'lowest': ('lo', 'w', 'e', 'st', '</w>'), 'new': ('n', 'e', 'w', '</w>'), 'newer': ('n', 'e', 'w', 'e', 'r', '</w>')}, ('n', 'e', 'w', '</w>', 'e', 's', 't', '</w>'))

**Result.** With the stated frequency table and deterministic tie break, the two learned merges and the resulting character-plus-`##` tokenization of `newest` are shown above. To display continuation pieces, the first subword is printed as-is and later ordinary pieces receive the `##` prefix; the end marker is removed for the human-readable answer. This variant uses the corpus frequencies exactly as they appear in the question.

In [8]:
def wordpiece_display(units):
    pieces = [u for u in units if u != "</w>"]
    return [piece if i == 0 else "##" + piece for i, piece in enumerate(pieces)]

newest_units = segmented["new"] + ("e", "s", "t", "</w>")
{"merges": merge_history, "newest tokens": wordpiece_display(newest_units)}

{'merges': [(('s', 't'), 3, 0.3333333333333333), (('l', 'o'), 10, 0.1)], 'newest tokens': ['n', '##e', '##w', '##e', '##s', '##t']}

## 3. Raw-count TF-IDF and sentence similarity

    Sentences are documents and `cat`, `dog`, `animal`, and `pet` are the vocabulary. TF is the raw count. Unsmoothed IDF is defined here as `ln(N / df)` (natural logarithm), a common unsmoothed convention. Cosine similarity compares the resulting sentence vectors; the diagonal self-similarities are omitted when finding the closest pair.

In [10]:
import math
import numpy as np
from itertools import combinations

docs = ["cat dog animal", "cat cat pet", "dog pet animal", "cat dog pet"]
vocab = ["cat", "dog", "animal", "pet"]
term_counts = np.array([[doc.split().count(term) for term in vocab] for doc in docs], dtype=float)
df = (term_counts > 0).sum(axis=0)
idf = np.log(len(docs) / df)
tfidf = term_counts * idf
cosine = (tfidf @ tfidf.T) / np.outer(np.linalg.norm(tfidf, axis=1), np.linalg.norm(tfidf, axis=1))
pairs = [(i, j, cosine[i, j]) for i, j in combinations(range(len(docs)), 2)]
{"vocabulary": vocab, "document frequencies": dict(zip(vocab, df.tolist())),
 "IDF": dict(zip(vocab, idf.round(4).tolist())),
 "TF-IDF rows": {f"S{i+1}": row.round(4).tolist() for i, row in enumerate(tfidf)},
 "cosine similarities": [(f"S{i+1}", f"S{j+1}", round(float(s), 4)) for i, j, s in pairs],
 "closest pair": max(pairs, key=lambda x: x[2])}

{'vocabulary': ['cat', 'dog', 'animal', 'pet'], 'document frequencies': {'cat': 3, 'dog': 3, 'animal': 2, 'pet': 3}, 'IDF': {'cat': 0.2877, 'dog': 0.2877, 'animal': 0.6931, 'pet': 0.2877}, 'TF-IDF rows': {'S1': [0.2877, 0.2877, 0.6931, 0.0], 'S2': [0.5754, 0.0, 0.0, 0.2877], 'S3': [0.0, 0.2877, 0.6931, 0.2877], 'S4': [0.2877, 0.2877, 0.0, 0.2877]}, 'cosine similarities': [('S1', 'S2', 0.3201), ('S1', 'S3', 0.8719), ('S1', 'S4', 0.4133), ('S2', 'S3', 0.1601), ('S2', 'S4', 0.7746), ('S3', 'S4', 0.4133)], 'closest pair': (0, 2, np.float64(0.871882070542168))}

**Interpretation.** Terms appearing in every sentence have zero IDF, so `cat` and `dog` contribute no weight here. The displayed pairwise cosine scores identify the closest two sentences under this exact weighting and distance convention. Because the question says unsmoothed IDF but does not specify a logarithm base, using another log base rescales all vectors by the same constant and leaves cosine similarities unchanged.

## 4. Bernoulli Naive Bayes for the all-zero email

    The features are binary word-presence indicators. Use maximum-likelihood estimates from the four spam and three not-spam messages, equal to the observed class priors, and no smoothing (as the question does not specify smoothing). For E8 = `(0, 0, 0)`, each class score is its prior multiplied by the probability of absence for each feature.

In [13]:
import numpy as np

X = np.array([[1,1,1],[1,1,0],[1,0,1],[0,1,1],
              [0,0,1],[0,1,0],[1,0,0]])
y = np.array(["Spam"]*4 + ["NotSpam"]*3)
x_new = np.array([0, 0, 0])
scores = {}
details = {}
for cls in ("Spam", "NotSpam"):
    rows = X[y == cls]
    prior = len(rows) / len(X)
    p_present = rows.mean(axis=0)
    likelihood = np.where(x_new == 1, p_present, 1 - p_present)
    score = prior * np.prod(likelihood)
    scores[cls] = score
    details[cls] = {"prior": prior, "P(feature present) [free, win, offer]": p_present.tolist(),
                     "P(E8 | class)": float(np.prod(likelihood)), "unnormalized posterior": float(score)}
posterior = {c: v / sum(scores.values()) for c, v in scores.items()}
{"class calculations": details, "normalized posterior": posterior,
 "prediction": max(scores, key=scores.get)}

{'class calculations': {'Spam': {'prior': 0.5714285714285714, 'P(feature present) [free, win, offer]': [0.75, 0.75, 0.75], 'P(E8 | class)': 0.015625, 'unnormalized posterior': 0.008928571428571428}, 'NotSpam': {'prior': 0.42857142857142855, 'P(feature present) [free, win, offer]': [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], 'P(E8 | class)': 0.2962962962962964, 'unnormalized posterior': 0.126984126984127}}, 'normalized posterior': {'Spam': np.float64(0.0656934306569343), 'NotSpam': np.float64(0.9343065693430658)}, 'prediction': 'NotSpam'}

**Interpretation.** The no-smoothing spam likelihood for E8 is zero because every spam training example contains at least one of the three words. Not-spam has a nonzero probability for an email with none of them, so E8 is classified as **NotSpam**. Laplace smoothing would avoid the zero likelihood, but would be an additional assumption not requested by the paper.

## 5. Interpolated Kneser-Ney bigram probabilities

    Count only the adjacent word pairs in the five sentences; no BOS/EOS tokens are introduced. With discount `d = 0.5`, use `P_KN(w|h) = max(c(h,w)-d, 0)/c(h) + lambda(h) P_cont(w)`, where `lambda(h) = d * N1+(h,*) / c(h)` and `P_cont(w) = N1+(*,w) / N1+(*,*)`. The continuation denominator is the number of distinct bigram types.

In [16]:
from collections import Counter, defaultdict

sentences = ["I like NLP", "I like AI", "You like NLP", "You love AI", "We love NLP"]
bigrams = Counter((a, b) for sentence in sentences for a, b in zip(sentence.split(), sentence.split()[1:]))
followers = defaultdict(set)
predecessors = defaultdict(set)
for (h, w), count in bigrams.items():
    followers[h].add(w)
    predecessors[w].add(h)
total_types = len(bigrams)
p_cont = {w: len(predecessors[w]) / total_types for w in predecessors}
discount = 0.5

def p_kn(word, history):
    history_count = sum(count for (h, _), count in bigrams.items() if h == history)
    if history_count == 0:
        return p_cont.get(word, 0.0)
    count = bigrams[(history, word)]
    first_term = max(count - discount, 0) / history_count
    backoff_weight = discount * len(followers[history]) / history_count
    return first_term + backoff_weight * p_cont.get(word, 0.0)

{"bigram counts": dict(bigrams), "distinct bigram types": total_types,
 "continuation probabilities": p_cont,
 "P(AI | love)": p_kn("AI", "love"), "P(NLP | like)": p_kn("NLP", "like")}

{'bigram counts': {('I', 'like'): 2, ('like', 'NLP'): 2, ('like', 'AI'): 1, ('You', 'like'): 1, ('You', 'love'): 1, ('love', 'AI'): 1, ('We', 'love'): 1, ('love', 'NLP'): 1}, 'distinct bigram types': 8, 'continuation probabilities': {'like': 0.25, 'NLP': 0.25, 'AI': 0.25, 'love': 0.25}, 'P(AI | love)': 0.375, 'P(NLP | like)': 0.5833333333333334}

**Interpretation.** `AI` has been preceded by both `like` and `love`, so its continuation probability is based on two distinct preceding contexts. `NLP` has also occurred after multiple contexts. The `love` history contains two distinct continuation types across two occurrences, while `like` occurs three times and has two distinct follower types. The numeric outputs show the discounted direct count plus the continuation backoff contribution for each requested probability.